# 頭皮 Grid 法向量 與 髮絲存取 示範（per-strand boolean mask 版）

這本 notebook 示範兩件事：

1. **讀取髮絲**：DiffLocks 資料集裡髮絲怎麼存、如何存取單根髮絲 / 一批髮絲（髮束）。
2. **頭皮 grid 法向量**：把頭皮切成 n×n grid 後，如何存取每個 grid cell 對應的 3D 位置與法向量。

跟原版 `scalp_grid_and_strands_demo.ipynb` 的差異在第 5、6 節：這裡改成把自訂的 n×n color matrix
直接轉成**逐髮絲 (per-strand)** 的 `highlighted` / `highlight_color`，存成
`coloring_by_strand/generate_highlight_templates_by_strand.py` 認得的 template 格式，並改呼叫
`coloring_by_strand/generate_highlight_render_by_strand.py` 渲染——不再是 `coloring_by_grid` 那種
「n×n grid template，渲染時逐格查表」的做法。

**本 notebook (`mask_updating_test.ipynb`) 是 `scalp_grid_and_strands_demo_by_strand.ipynb` 的複本**：所有 RGB 顏色矩陣
（n×n 的 `color_matrix`、逐髮絲的 `strand_colors`、逐格的 `grid_color` 等）都改成 **boolean 矩陣**，
`False (0)` = base color、`True (1)` = highlight color。只有在畫圖 / 存 template 給 Blender 時，
才用 `mask_to_rgb()` 把 mask 換回 RGB。

用到的函式定義在同資料夾的 `scalp_grid.py`、`hair_query.py`、`ply_io.py`（在前面對話中建立），
以及 `../scalp_uv_grid.py`（`coloring_by_strand` pipeline 共用的 UV<->grid 換算）。

**本 notebook (`mask_updating_test_3D.ipynb`) 是 `mask_updating_test.ipynb` 的複本，差異只在「散度 (divergence) 的算法」**：
原版先把每格內的 3D 髮絲流向投影到 2D (t1, t2 平面) 再算散度 (勾邊測試、active contour 的邊緣圖都是)；
這個版本改成**直接在 3D 空間算散度**——每格用 3D 取樣點 / 3D 流向向量 (t1, t2, normal 局部座標) 做 3D affine 最小平方擬合，
div = ∂vx/∂x + ∂vy/∂y + ∂vh/∂h (= 3×3 Jacobian 的 trace)；**只有畫圖 (quiver / 稠密流向場) 才把 3D 向量投影到 2D**。
Active contour 的邊緣圖 `|div v|` 也改成直接吃每格的 3D 散度，不再由 2D 稠密場用 `np.gradient` 求導；
snake 本身也直接在 3D 頭皮曲面上跑 (contour 是 3D 點列，吸附 / 錨定 / 弧長都用 3D 距離，每步投影回曲面)，只有輸出 N×N mask 與畫 2D 圖時才對應回格子座標。
輸出的 template / render 也另存到 `*_3D` 資料夾，不會蓋掉原 notebook 的結果。

**能量定義再修改：`|div v|` -> `|flux|`**。邊緣訊號 (勾邊測試熱力圖、active contour 的邊緣圖) 不再是 affine 擬合出來的散度，
改成散度定理邊界形式的**淨向外通量** ∮ v·n̂ dA：對每格 3D 柱體 (±x, ±y, ±h 三對面)，直接在靠近各面的髮絲取樣點上量單位流向的法向分量
(`edge_detect_flux_3d`)，不擬合、不求導；再除以柱體尺度得到無因次的單位體積通量 (跟散度同尺度)。

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 這個 notebook 放在 .../highlighting/generate_from_3D_models/scalp_grid_volume_sample/ 底下,
# 但同資料夾的模組是用「package 形式」匯入
# (from highlighting.generate_from_3D_models.scalp_grid_volume_sample.xxx import ...),
# 所以要把專案根目錄 (P76154862) 加進 sys.path,Python 才找得到 `highlighting` 這個套件。
HERE = "/home/kyh/Desktop/P76154862/highlighting/generate_from_3D_models/scalp_grid_volume_sample"
PROJECT_ROOT = os.path.abspath(os.path.join(HERE, "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from highlighting.generate_from_3D_models.scalp_grid_volume_sample.scalp_grid import (
    load_scalp_mesh, build_scalp_grid, build_scalp_grid_tangents, nearest_valid_cell,
)
from highlighting.generate_from_3D_models.scalp_grid_volume_sample.hair_query import (
    load_strands, strands_above_cell, strands_above_cells, strands_above_cells_gpu,
    root_density_map, estimate_half_size,
)
from highlighting.generate_from_3D_models.scalp_grid_volume_sample.tool_functions import (
    show_color_matrix, draw_strand_uv_color_distribution, apply_median_color_to_strands,
    save_template_npz, run_blender_render,
)

print("PROJECT_ROOT =", PROJECT_ROOT)

## 1. 讀取頭皮 mesh，建立 n×n grid

In [ ]:
# main.py 裡 DEFAULT_DATASET_ROOT 是用 os.path.dirname(HERE) 推算,
# 但這個資料夾被搬進 highlighting/ 專案後那個預設值已經不對了 (會指到 generate_from_3D_models/),
# 這裡直接指到實際的 DiffLocks_Dataset 位置。
DATASET_ROOT = os.path.join(PROJECT_ROOT, "DiffLocks_Dataset", "difflocks_dataset")
SCALP_PLY = os.path.join(DATASET_ROOT, "body_data", "scalp.ply")

positions, normals, uv, faces = load_scalp_mesh(SCALP_PLY)
print(f"scalp mesh: {positions.shape[0]} vertices, {faces.shape[0]} faces")

N = 96  # grid 解析度 (n x n)
grid_uv, grid_pos, grid_normal, grid_valid, (us, vs) = build_scalp_grid(
    positions, normals, uv, faces, N
)

# 每格的切線基底 (t1, t2)：t1 沿 UV 的 +u、t2 = normal x t1 沿 +v，(t1, t2, normal) 是右手正交基，
# 而且相鄰格連續。整個 notebook 所有用到切線的地方 (柱體篩選、2D 投影、向量場、分數) 都用這一組，
# 舊的 tangent_frame(normal) 已經移除，所有函式都強制要求傳入 grid_t1 / grid_t2。
grid_t1, grid_t2 = build_scalp_grid_tangents(positions, normals, uv, faces, grid_normal, grid_valid)

print(f"grid_pos.shape    = {grid_pos.shape}     # (N, N, 3) 每格在頭皮表面的 3D 位置")
print(f"grid_normal.shape = {grid_normal.shape}  # (N, N, 3) 每格的法向量")
print(f"grid_t1/t2.shape  = {grid_t1.shape}  # (N, N, 3) 每格的切線基底 (t1: +u, t2: +v，皆與法向量正交)")
print(f"grid_valid.shape  = {grid_valid.shape}   # (N, N) bool, 是否落在頭皮的 UV 範圍內")
print(f"有效 (落在頭皮上) 的 cell 數: {grid_valid.sum()} / {grid_valid.size}")

## 2. 存取單一 grid cell 的法向量

`grid_pos` / `grid_normal` 都是規則的 `(N, N, 3)` array，直接用 `[i, j]` 索引就能拿到該格的位置與法向量——
不需要額外查找。若 `(i, j)` 不在頭皮上（`grid_valid[i, j] == False`），用 `nearest_valid_cell` 找最近的有效格。

In [ ]:
i, j = N // 2, N // 2
i, j = nearest_valid_cell(grid_valid, i, j)

cell_pos    = grid_pos[i, j]     # (3,) 這一格在頭皮表面的 3D 位置
cell_normal = grid_normal[i, j]  # (3,) 這一格的法向量 (已正規化)

print(f"grid cell ({i}, {j})")
print("  位置 (position):", cell_pos)
print("  法向量 (normal):", cell_normal, " |n| =", np.linalg.norm(cell_normal))

In [ ]:
# 視覺化整個 grid 的法向量分布 (紅色箭頭), 黑點 = 上面查詢的 cell
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")

valid_pos = grid_pos[grid_valid]
ax.scatter(valid_pos[:, 0], valid_pos[:, 2], valid_pos[:, 1], s=3, alpha=0.35, color="steelblue")

stride = 2
sv = grid_valid[::stride, ::stride]
sp = grid_pos[::stride, ::stride][sv]
sn = grid_normal[::stride, ::stride][sv]
ax.quiver(sp[:, 0], sp[:, 2], sp[:, 1], sn[:, 0], sn[:, 2], sn[:, 1], length=0.02, color="crimson", linewidth=0.7)

ax.scatter([cell_pos[0]], [cell_pos[2]], [cell_pos[1]], color="black", s=60, label=f"cell ({i},{j})")
ax.set_xlabel("x"); ax.set_ylabel("z"); ax.set_zlabel("y")
ax.set_title(f"scalp {N}x{N} grid normals")
ax.legend()
plt.show()

## 3. 讀取髮絲資料

In [ ]:
HAIRSTYLE_DIR = os.path.join(DATASET_ROOT, "generated_hairstyles", "base_13_idx_94847")
STRANDS_NPZ = os.path.join(HAIRSTYLE_DIR, "full_strands.npz")

strand_positions, root_uv, root_normal = load_strands(STRANDS_NPZ)
print("strand_positions:", strand_positions.shape, strand_positions.dtype)  # (S, P, 3)
print("root_uv         :", root_uv.shape)   # (S, 2)  每根髮絲髮根在頭皮 UV 上的座標
print("root_normal     :", root_normal.shape)  # (S, 3)  髮根法向量

RGB=os.path.join(HAIRSTYLE_DIR, 'rgb.png')
img = plt.imread(RGB)
plt.figure(figsize=(6, 6))
plt.imshow(img)

### 3.1 單根髮絲存取

直接用整數索引即可：`positions[k]` 拿到整條折線，`root_uv[k]` / `root_normal[k]` 是它的髮根資訊。

In [ ]:
k = 12345
strand_k = strand_positions[k]  # (P, 3): 這根髮絲所有取樣點 (P=256)

print(f"strand[{k}].shape = {strand_k.shape}")
print("  髮根 (第0點):", strand_k[0])
print("  髮尾 (末點) :", strand_k[-1])
print("  root_uv     :", root_uv[k])
print("  root_normal :", root_normal[k])

seg_len = np.linalg.norm(np.diff(strand_k, axis=0), axis=1).sum()
print(f"  髮絲長度 (折線段長度加總) ≈ {seg_len:.4f}")

### 3.2 一批髮絲（髮束）：拿 grid cell 的位置 + 法向量，找出「這一格正上方」的髮絲

資料本身沒有內建的髮束/群組 id，所以用第 2 節拿到的 `cell_pos` / `cell_normal` 建立一個沿法向量延伸的局部柱體，
測試每根髮絲的取樣點是否落在柱體內（`strands_above_cell`），藉此把「頭皮某一格」跟「它上面的一撮頭髮」連結起來。

In [ ]:
t1, t2 = grid_t1[i, j], grid_t2[i, j]             # 該 cell 的切線方向 (與法向量共同組成局部座標系，跟相鄰格對齊)
half_size = estimate_half_size(grid_pos, grid_valid)  # 柱體的橫截面半徑 (預設用 grid 間距估計)

strand_mask, point_mask, local_h = strands_above_cell(
    strand_positions, cell_pos, cell_normal, t1, t2,
    half_size=half_size, h_min=-0.01, h_max=0.4,
)
bundle_idx = np.where(strand_mask)[0]
bundle = strand_positions[bundle_idx]  # (K, P, 3): 這一整束髮絲

print(f"grid cell ({i}, {j}) 上方共有 {bundle_idx.size} 根髮絲")
print("前 10 個 strand index:", bundle_idx[:10])
print("bundle.shape =", bundle.shape)

## 5. 自訂 n×n Mask Matrix，直接轉成逐髮絲 (per-strand) mask

延續第 1 節「頭皮 UV → n×n grid」的做法，一樣手動自訂一個 n×n 的 boolean 矩陣（每一格 `False` = base color、`True` = highlight color），
鋪在頭皮 UV 空間上；但這次不是把矩陣存成「grid template」讓 Blender 在渲染時逐格查表（`coloring_by_grid`
的做法），而是**在 notebook 這裡就把每一根髮絲的 mask 算好**，變成 `highlighted` / `highlight_color`
這兩個逐髮絲 (per-strand) 的 array —— 跟 `coloring_by_strand/generate_highlight_templates_by_strand.py`
產生的 template 是同一種資料格式。

UV → grid 的 row/col 換算直接呼叫 `scalp_uv_grid.py` 的 `build_uv_bin_edges` / `uv_to_grid_rowcol` /
`grid_cell_centers_rowcol`，跟 `coloring_by_strand` pipeline 用的是同一套慣例（row 0 = UV 高 v 端，
跟其他 template PNG / `scalp_grid_rgb.jpg` 一致），不再像舊版那樣自己用 `searchsorted` 土法煉鋼。

In [ ]:
from highlighting.generate_from_3D_models.scalp_uv_grid import (
    build_uv_bin_edges, grid_cell_centers_rowcol, load_scalp_uv, uv_to_grid_rowcol,
)

DATASET_PATH = DATASET_ROOT  # generate_from_3D_models pipeline 慣用的名稱 (含 generated_hairstyles/, body_data/)

scalp_uv = load_scalp_uv(DATASET_PATH)
us, vs = build_uv_bin_edges(scalp_uv, N)

row_idx, col_idx = uv_to_grid_rowcol(root_uv, us, vs)


strands_mask_grid = strands_above_cells_gpu(
    strand_positions, grid_pos, grid_normal, half_size,
    h_min=-0.01, h_max=0.4, grid_valid=grid_valid, show_progress=True,
    grid_t1=grid_t1, grid_t2=grid_t2,
)



## 勾邊測試

In [ ]:
from tqdm import tqdm
from evaluation import project_strands_batch


def cell_flow_vectors_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size):
    """這個 grid cell 裡每根髮絲的 (取樣位置, 流向向量)，**全部留在 3D**，不做任何 2D 投影。

    座標用該格局部基底 (t1, t2, normal) 表示、以格子中心 pos 為原點 -> 每個點是 (x, y, h)：
    x, y 沿 UV 的 +u / +v 切線，h 沿法向量 (離頭皮的高度)。每根髮絲取「落在這格柱體內的第一個 / 最後一個取樣點」
    當 start / end，取樣位置 = 中點、流向 = end - start (未正規化)。t1, t2 來自 build_scalp_grid_tangents，
    相鄰格連續，所以不同格的 (x, y, h) 對應到一致的方向。
    回傳 (K', 3) 取樣位置、(K', 3) 流向向量 (只含柱體內至少有一個取樣點的髮絲)。
    """
    strands_inside = np.asarray(strands_inside)
    if strands_inside.size == 0:
        return np.zeros((0, 3)), np.zeros((0, 3))

    batch = strand_positions[strands_inside]  # (K, P, 3)
    mask, _, _ = project_strands_batch(batch, pos, t1, t2, normal, half_size)  # 只借它的柱體 mask
    has_valid = mask.any(axis=1)
    if not has_valid.any():
        return np.zeros((0, 3)), np.zeros((0, 3))

    basis = np.stack([t1, t2, normal], axis=1)  # (3, 3): 世界座標 -> (x, y, h) 局部座標
    local = (batch - pos[None, None, :]) @ basis  # (K, P, 3)，3D 局部座標 (含 h 分量，不丟掉)

    first_idx = mask.argmax(axis=1)
    last_idx = mask.shape[1] - 1 - mask[:, ::-1].argmax(axis=1)
    rows = np.arange(strands_inside.shape[0])
    starts = local[rows, first_idx]  # (K, 3)
    ends = local[rows, last_idx]     # (K, 3)

    sample_pts = (starts + ends) * 0.5  # 向量場的取樣位置: 每根髮絲的中點
    directions = ends - starts          # 每根髮絲的流向 (未正規化)
    return sample_pts[has_valid], directions[has_valid]


def cell_points_tangents_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size, h_min=-0.01, h_max=0.4):
    """這個 grid cell 柱體內「所有」髮絲取樣點的 3D 局部座標 (x, y, h) 與該點的單位切線 (流向)。
    座標基底跟 cell_flow_vectors_3d 一樣是 (t1, t2, normal)，以格子中心為原點；切線用相鄰取樣點的中央差分再正規化。
    回傳 pts (M, 3), tans (M, 3)。"""
    strands_inside = np.asarray(strands_inside)
    if strands_inside.size == 0:
        return np.zeros((0, 3)), np.zeros((0, 3))

    batch = strand_positions[strands_inside]  # (K, P, 3)
    mask, _, _ = project_strands_batch(batch, pos, t1, t2, normal, half_size, h_min=h_min, h_max=h_max)
    if not mask.any():
        return np.zeros((0, 3)), np.zeros((0, 3))

    basis = np.stack([t1, t2, normal], axis=1)
    local = (batch - pos[None, None, :]) @ basis                                    # (K, P, 3)
    tang = np.gradient(local, axis=1)                                               # (K, P, 3) 沿髮絲的中央差分
    tang = tang / (np.linalg.norm(tang, axis=-1, keepdims=True) + 1e-12)
    return local[mask], tang[mask]


def edge_detect_flux_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size,
                        band=0.3, h_min=-0.01, h_max=0.4, min_pts=3):
    """(局部) 3D 向量場穿過這格柱體邊界的「淨向外通量」(有號)，回傳 (flux, ok)。

    通量取自散度定理的邊界形式: flux = ∮ v · n̂ dA。柱體 (x, y ∈ ±half_size, h ∈ [h_min, h_max]) 有三對面
    (±x, ±y, ±h)；v 用每個取樣點的單位流向 (cell_points_tangents_3d)，直接在邊界附近的取樣點上量法向分量，
    **不做任何最小平方擬合、也不求導**：
      * 每個面取靠近該面、佔該軸全長 band 比例的取樣點 (band 內)，v·n̂ 的平均 = 這個面上的通量密度 (n̂ = 向外法向量)
      * 一對面的淨向外通量密度 = (+面的 <v_a>) - (-面的 <v_a>)
      * 除以兩個 band 中心的距離 D_a、再乘上 half_size -> 無因次 (單位體積的淨通量，跟散度同尺度，所以熱力圖的
        色階 / 門檻大致沿用；柱體體積每格相同，所以跟總通量只差一個常數倍)
    三對面 (x, y, h) 的貢獻相加。某一軸任一側的 band 內取樣點 < min_pts (例如髮絲到不了柱體頂端) 就略過那一軸；
    三軸都略過時 ok = False (flux = 0)。
    這格裡 c1 / c2 兩類髮絲流向一致 (沒有邊界) -> 流進 = 流出，淨通量 ≈ 0；顏色邊界 / 分邊剛好穿過 (髮絲往兩側散開或從兩側
    匯合) -> 淨通量絕對值變大。
    """
    pts, tans = cell_points_tangents_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size, h_min, h_max)
    if pts.shape[0] < 2 * min_pts:
        return 0.0, False

    half_ext = np.array([half_size, half_size, 0.5 * (h_max - h_min)])
    center = np.array([0.0, 0.0, 0.5 * (h_max + h_min)])
    u = (pts - center) / half_ext                  # 各軸正規化到 [-1, 1]，±1 就是柱體的面
    edge = 1.0 - 2.0 * band                        # u >= edge 的點屬於 + 面的 band，u <= -edge 屬於 - 面的 band

    flux, used = 0.0, 0
    for a in range(3):
        hi, lo = u[:, a] >= edge, u[:, a] <= -edge
        if hi.sum() < min_pts or lo.sum() < min_pts:
            continue
        net_out = tans[hi, a].mean() - tans[lo, a].mean()   # (v·n̂ at +a 面) + (v·n̂ at -a 面, n̂ = -e_a)
        d_centers = 2.0 * (1.0 - band) * half_ext[a]        # 兩個 band 中心的距離
        flux += net_out * half_size / d_centers
        used += 1
    return (float(flux), True) if used else (0.0, False)


def edge_detect_score(strands_inside, strand_positions, pos, normal, t1, t2, half_size, is_c1_mask):
    """邊緣分數 = |淨向外通量| (直接在 3D 局部座標算，見 edge_detect_flux_3d)。
    is_c1_mask 已經不使用 (保留參數只是為了跟原版簽名一致)。"""
    flux, _ = edge_detect_flux_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size)
    return abs(flux)


def cal_grid_flux_3d(
    strand_positions, grid_pos, grid_normal, grid_valid, half_size, strand_mask_grid,
    *, grid_t1, grid_t2,
):
    """對每個 grid cell 算 3D 淨向外通量 (有號)，排成跟 mask_matrix 一致的 (row, col) 排列。
    回傳 (flux_rc, ok_rc)，皆為 (N, N)：ok_rc = 這格邊界附近取樣點夠多、通量真的有算出來。"""
    n_rows, n_cols = grid_valid.shape
    flux_rc = np.zeros((n_rows, n_cols), dtype=np.float32)
    ok_rc = np.zeros((n_rows, n_cols), dtype=bool)
    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        if not grid_valid[u_idx, v_idx]:
            continue
        row, col = (n_cols - 1) - v_idx, u_idx  # (u_idx, v_idx) -> (row, col)
        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]
        flux_rc[row, col], ok_rc[row, col] = edge_detect_flux_3d(
            strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx],
            grid_t1[u_idx, v_idx], grid_t2[u_idx, v_idx], half_size,
        )
    return flux_rc, ok_rc


def cal_grid_edge_score(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, strand_mask, half_size, base_color,
    strand_mask_grid = None,
    *, grid_t1, grid_t2,  # (N, N, 3) 每格切線基底 (必填)，用 build_scalp_grid_tangents 產生
):
    n_rows, n_cols = grid_valid.shape

    if strand_mask_grid is None:
        strand_mask_grid = strands_above_cells_gpu(
            strand_positions, grid_pos, grid_normal, half_size,
            h_min=-0.01, h_max=0.4, grid_valid=grid_valid, show_progress=True,
            grid_t1=grid_t1, grid_t2=grid_t2,
        )

    grid_score = np.zeros((n_rows, n_cols), dtype=np.float32)  # (N, N) 每格的邊緣分數 (越大 = 顏色邊界越明顯)
    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        row, col = (n_cols - 1) - v_idx, u_idx  # (u_idx, v_idx) -> (row, col)，見上面說明

        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]  # (K,) 這一格上方的髮絲索引
        grid_score[row, col] = edge_detect_score(
            strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx],
            grid_t1[u_idx, v_idx], grid_t2[u_idx, v_idx], half_size, None
        )
    return grid_score


plt.figure(figsize=(8, 6 ))

grid_score = cal_grid_edge_score(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, None, half_size, None,
    strand_mask_grid=strands_mask_grid,
    grid_t1=grid_t1, grid_t2=grid_t2,
)

print("average grid score =", grid_score.mean())

# plt.subplot(2, (n_iter+1)//2+1, i+1)

plt.imshow(grid_score, origin="upper", cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="3D 向量場通量 |flux| (越大 = 顏色邊界越明顯)")
plt.title(f"grid score (iteration {i})")

plt.show()

threshold = 0.7

# grid_score_mask = grid_score < threshold
# grid_score[grid_score_mask] = 0.0


plt.imshow(grid_score, origin="upper", cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="3D 向量場通量 |flux| (越大 = 顏色邊界越明顯)")

In [ ]:
import cv2
from scipy import ndimage as ndi

# 在通量圖 (grid_score = |flux|, 排列同 mask_matrix 的 (row, col)) 上做 Canny 邊緣偵測
div_map = grid_score.copy()

# (u, v) -> (row, col) 排列的有效格遮罩；頭皮外的格子分數是 0，會在頭皮邊界造成假邊緣，所以之後要排除
valid_rc = grid_valid.T[::-1]
valid_inner = ndi.binary_erosion(valid_rc, iterations=2)  # 往內縮 2 格，避開頭皮邊界

# Canny 吃 uint8: 用跟上面 imshow 一樣的 [0, vmax] 範圍線性映射到 0~255
vmax = 1.0
div_u8 = (np.clip(div_map / vmax, 0, 1) * 255).astype(np.uint8)
div_u8 = cv2.GaussianBlur(div_u8, (3, 3), sigmaX=0.8)  # 輕微平滑，抑制單格雜訊

canny_low, canny_high = 40, 120  # 遲滯門檻 (hysteresis): >high 一定是邊緣, low~high 只有連到強邊緣才保留
edges = cv2.Canny(div_u8, canny_low, canny_high, apertureSize=3, L2gradient=True) > 0
edges &= valid_inner

# 只做 Sobel 濾波 (不含 blur / 非極大值抑制 / 遲滯門檻)：直接對通量圖算 x, y 方向梯度，再取梯度大小
sobel_x = cv2.Sobel(div_map.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(div_map.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
sobel_mag = np.hypot(sobel_x, sobel_y)

print(f"edge cells = {edges.sum()} / {valid_rc.sum()} valid cells")

fig, axes = plt.subplots(4,1, figsize=(8, 32))

axes[0].imshow(div_map, origin="upper", cmap="magma", vmin=0, vmax=vmax)
axes[0].set_title("flux map |flux|")

im = axes[1].imshow(sobel_mag, origin="upper", cmap="viridis")
axes[1].set_title("Sobel gradient magnitude only (ksize=3)")
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(edges, origin="upper", cmap="gray")
axes[2].set_title(f"Canny edges (low={canny_low}, high={canny_high})")

axes[3].imshow(div_map, origin="upper", cmap="magma", vmin=0, vmax=vmax)
overlay = np.zeros((*edges.shape, 4))
overlay[edges] = (0.0, 1.0, 1.0, 1.0)  # 邊緣格用青色疊在通量圖上
axes[3].imshow(overlay, origin="upper")
axes[3].set_title("edges overlaid on flux map")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:

def cell_flow_vectors(strands_inside, strand_positions, pos, normal, t1, t2, half_size, is_c1_mask):
    """跟通量 (edge_detect_flux_3d) 同一批 3D 髮絲流向 (cell_flow_vectors_3d)，這裡只是把 (x, y, h) 的 h 分量
    丟掉 -> 2D (x, y)，**純粹給畫圖 (quiver) 用的投影**；通量本身完全不經過這個函式。"""
    pts3, vecs3 = cell_flow_vectors_3d(strands_inside, strand_positions, pos, normal, t1, t2, half_size)
    return pts3[:, :2], vecs3[:, :2]


def compute_grid_vector_field(
    strand_positions, grid_pos, grid_normal, grid_valid, half_size,
    strand_mask, strand_mask_grid, grid_t1, grid_t2,
):
    """對每個有效 grid cell，把 cell_flow_vectors_3d 算出的 (取樣點, 流向向量) 取平均當作
    這一格的「代表向量」(局部 3D，除以 half_size 變成無因次尺度)，再轉回 3D world space
    (mean_x * t1 + mean_y * t2 + mean_h * normal) 方便疊在頭皮上畫 quiver。

    這個平均向量的長度，跟 edge_detect_score 算通量用的是同一個流向場的另一種讀法：
    c1/c2 兩類髮絲流向大致一致 -> 平均後箭頭長；兩類方向相反、在此交錯抵消 -> 平均後箭頭
    變短甚至消失 (代表這裡場不連續，|flux| 容易偏大)。

    t1/t2 = grid_t1 / grid_t2 (build_scalp_grid_tangents 算好的、跟 grid 對齊的切線基底，
    整個 grid 只算一次，每格重複使用)。

    註: 這個函式保留在這裡供參考/其他用途，目前下面的視覺化已經改用不平均的
    compute_strand_vectors_2d，不再呼叫這支函式。"""
    n_rows, n_cols = grid_valid.shape
    is_c1_mask = ~strand_mask  # False = base color = c1

    grid_vec3d = np.zeros((n_rows, n_cols, 3), dtype=np.float32)
    grid_vec_mag = np.zeros((n_rows, n_cols), dtype=np.float32)

    for u_idx, v_idx in np.ndindex(n_rows, n_cols):
        if not grid_valid[u_idx, v_idx]:
            continue
        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]
        cell_pos_ij = grid_pos[u_idx, v_idx]
        cell_normal_ij = grid_normal[u_idx, v_idx]
        t1 = grid_t1[u_idx, v_idx]
        t2 = grid_t2[u_idx, v_idx]

        _, vecs = cell_flow_vectors_3d(
            strands_inside, strand_positions, cell_pos_ij, cell_normal_ij, t1, t2, half_size
        )
        if vecs.shape[0] == 0:
            continue

        mean_vec = vecs.mean(axis=0) / half_size  # (3,) 無因次化，方便固定 quiver 顯示尺度
        grid_vec3d[u_idx, v_idx] = mean_vec[0] * t1 + mean_vec[1] * t2 + mean_vec[2] * cell_normal_ij
        grid_vec_mag[u_idx, v_idx] = np.linalg.norm(mean_vec)

    return grid_vec3d, grid_vec_mag


def compute_strand_vectors_2d(
    strand_positions, grid_pos, grid_normal, grid_valid, half_size,
    strand_mask, base_color, strand_mask_grid, grid_t1, grid_t2,
):
    """**只用於視覺化**：把 3D 流向場 (cell_flow_vectors_3d，通量就是直接在這個 3D 場上算) 投影到 (t1, t2)
    平面 (丟掉法向量 h 分量)，跟通量計算無關。除以 half_size 無因次化，但**不**把每格內所有髮絲的向量平均成一支代表箭頭，而是保留每一根
    髮絲自己的 (取樣位置, 流向向量)，攤平成 (M, 2) / (M, 2) 兩個陣列回傳，M = 所有有效格子
    裡的髮絲總數。用來疊圖時以類似 scatter 的方式，把每一根髮絲的流向都個別畫出來，而不是
    每格只看得到一支被平均、甚至互相抵消後消失的箭頭。

    座標轉換跟 compute_grid_vector_field_2d / cal_grid_score 用同一套對應方式：
    col = u_idx (沿 u 增加方向)，row = (n_cols - 1) - v_idx (沿 v 增加方向，但 row 是反過來
    的)。cell_flow_vectors 回傳的取樣位置/向量是在 (t1, t2) 局部座標系、以格子中心為原點、
    數值大約落在 [-half_size, half_size] 內，所以：
      - 取樣位置：除以 half_size 再乘 0.5，變成「相對格子中心的偏移，最多到格子邊界」，
        加到 (col, row) 上就會落在對應格子範圍內，畫出來才會跟 grid_score 熱力圖對齊。
      - 流向向量：除以 half_size 做無因次化 (跟 3D/平均版本一致)，local_y 分量因為 row 方向
        跟 v 方向相反，要取負號，箭頭方向才會跟 row/col 排列一致。

    t1/t2 = grid_t1 / grid_t2：build_scalp_grid_tangents 算好的、跟 grid 對齊的切線基底，整個 grid 只算一次。
    """
    n_rows, n_cols = grid_valid.shape
    # is_c1_mask = ~strand_mask

    all_pts, all_vecs = [], []

    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols)):
        if not grid_valid[u_idx, v_idx]:
            continue
        row, col = (n_cols - 1) - v_idx, u_idx  # 跟 cal_grid_score 同一套 (u_idx, v_idx) -> (row, col)

        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]
        sample_pts, vecs = cell_flow_vectors(
            strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx],
            grid_t1[u_idx, v_idx], grid_t2[u_idx, v_idx], half_size, None,
        )
        if vecs.shape[0] == 0:
            continue

        pts_plot = np.stack([
            col + 0.5 * sample_pts[:, 0] / half_size,
            row - 0.5 * sample_pts[:, 1] / half_size,
        ], axis=1)
        vecs_plot = np.stack([
            vecs[:, 0] / half_size,
            -vecs[:, 1] / half_size,
        ], axis=1)

        all_pts.append(pts_plot)
        all_vecs.append(vecs_plot)

    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)

    return np.concatenate(all_pts, axis=0).astype(np.float32), np.concatenate(all_vecs, axis=0).astype(np.float32)


strand_pts2d, strand_vec2d = compute_strand_vectors_2d(
    strand_positions, grid_pos, grid_normal, grid_valid, half_size,
    None, None, strands_mask_grid, grid_t1, grid_t2,
)
print("strand_pts2d:", strand_pts2d.shape)  # (M, 2) M = 有效格子裡的髮絲取樣點總數 (未平均)

# 未平均的話箭頭數 = 髮絲數，全部畫出來可能太密看不清、也畫得慢，固定隨機種子做子取樣。
max_arrows = 6000
if strand_pts2d.shape[0] > max_arrows:
    rng = np.random.default_rng(0)
    pick = rng.choice(strand_pts2d.shape[0], size=max_arrows, replace=False)
    plot_pts, plot_vecs = strand_pts2d[pick], strand_vec2d[pick]
else:
    plot_pts, plot_vecs = strand_pts2d, strand_vec2d

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(grid_score, origin="upper", cmap="magma", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="3D 向量場通量 |flux| (越大 = 顏色邊界越明顯)")

# local_x 對齊 u 增加方向 = col 增加方向；local_y 對齊 v 增加方向，但 row = (n_cols-1) - v_idx
# 是反過來的，compute_strand_vectors_2d 內部已經把位置/向量的 y 分量都處理成跟 row 方向一致，
# 這裡直接畫就好，不用再額外取負號。
ax.quiver(
    plot_pts[:, 0], plot_pts[:, 1],
    plot_vecs[:, 0], plot_vecs[:, 1],
    color="cyan", scale=20, width=0.0015, alpha=0.5,
)
ax.set_xlim(-0.5, N - 0.5)
ax.set_ylim(N - 0.5, -0.5)  # 跟 imshow(origin="upper") 同方向: row 由上往下增加
ax.set_title("3D 通量 grid_score 熱力圖 + 每根髮絲 3D 流向投影到 2D 的向量 (cyan，僅視覺化，未平均)\n箭頭方向一致的區域 = 流向連續；箭頭雜亂交錯的區域 = 對應高 |flux|")
plt.show()

### 5.1 把每根髮絲的髮根對應到矩陣格子 -> 逐髮絲 highlighted / highlight_color

用 `scalp_uv_grid.build_uv_bin_edges` 對頭皮 mesh 的 UV bounding box 切出 `N` 個 bin
（跟 `coloring_by_strand/generate_highlight_templates_by_strand.py` 完全同一套函式，`N` 沿用第 1 節的定義），
再用 `uv_to_grid_rowcol` 把每根髮絲的 `root_uv`（第 3 節 `load_strands` 讀出的）對應到 `mask_matrix`
的 `(row, col)`，直接查表得到這根髮絲的 mask (bool)——每根髮絲一個顏色，不再是「一個 grid cell 對應一批髮絲
共用一個顏色，渲染時才查表」。

這裡示範「整顆頭都套用矩陣顏色」（`highlighted` 全開），等同於 `coloring_by_strand` pipeline 裡
`highlighted=True` 且 `highlight_start=0.0` 的效果——每根髮絲從髮根開始就是它查到的顏色。

In [ ]:
strand_highlighted = np.ones(root_uv.shape[0], dtype=bool)

# 每個 boolean 矩陣裡 False (0) = base_color、True (1) = highlight_color。
# 底下兩個 RGB 常數只用在「畫圖」與「存 template 給 Blender」時，把 mask 換回 RGB (mask_to_rgb)。
base_color = np.array([0.99, 0.99, 0.99])
highlight_color = np.array([0.0, 0.0, 0.0])


def mask_to_rgb(mask):
    """boolean mask (任意 shape) -> RGB float32 (shape + (3,))：False -> base_color, True -> highlight_color"""
    return np.where(np.asarray(mask, dtype=bool)[..., None], highlight_color, base_color).astype(np.float32)


def grid_mask_to_strand_mask(grid_mask, row_idx, col_idx):
    """(N, N) boolean 格子 mask (mask_matrix / ac_mask / cal_2d_grid_color 的輸出...) -> (S,) 逐髮絲 boolean mask，
    格式跟 base_strand_mask 完全一樣 (可直接放進 iter_strand_masks、餵給 mask_to_rgb / save_template_npz)。

    每根髮絲的 mask = 它髮根所在格子的值: strand_mask[k] = grid_mask[row_idx[k], col_idx[k]]。
    row_idx / col_idx 是 uv_to_grid_rowcol(root_uv, us, vs) 的結果 (第 5 節)，(row, col) 排列跟 mask_matrix 一致，
    所以 grid_mask 必須是 (row, col) 排列、shape = (N, N) (不是 grid_valid 那種 (u_idx, v_idx) 排列)。
    """
    grid_mask = np.asarray(grid_mask)
    if grid_mask.ndim != 2:
        raise ValueError(f"grid_mask 必須是 (N, N) 的 2D 陣列，收到 shape = {grid_mask.shape}")
    if row_idx.max() >= grid_mask.shape[0] or col_idx.max() >= grid_mask.shape[1]:
        raise ValueError(f"grid_mask shape {grid_mask.shape} 跟 row_idx / col_idx 的格子範圍不符")
    return grid_mask.astype(bool)[row_idx, col_idx]  # (S,) bool


mask_matrix = np.zeros((N, N), dtype=bool)  # (N, N) bool, 全 False = 全部 base_color

# 也可以改成手動逐格指定，例如只想染前兩格、其餘維持底色，取消註解試試：
# mask_matrix[5:10, 5:10] = True
# mask_matrix[5:10, -10:-5] = True
mask_matrix[2:15*3, 2:15*3] = True

show_color_matrix(mask_to_rgb(mask_matrix))


base_strand_mask = grid_mask_to_strand_mask(mask_matrix, row_idx, col_idx)  # (S,) bool: 每根髮絲自己的 mask，直接查表得到

print("base_strand_mask.shape      =", base_strand_mask.shape, base_strand_mask.dtype)
print("strand_highlighted.shape =", strand_highlighted.shape)
print("strand", k, "落在矩陣格子", (int(row_idx[k]), int(col_idx[k])), "-> mask", base_strand_mask[k])


In [ ]:
from tqdm import tqdm
from evaluation import *

def update_color(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, strand_mask, half_size,
    strand_mask_grid = None,
    threshold = 0.8,
    majority_ratio_threshold = 0.9,
    *, grid_t1, grid_t2,  # (N, N, 3) 每格切線基底 (必填)，用 build_scalp_grid_tangents 產生
):
    n_rows, n_cols = grid_valid.shape

    strand_mask_iter = strand_mask.copy()  # (S,) bool 每根髮絲自己的 mask, 會在迴圈裡被改掉
    if strand_mask_grid is None:
        strand_mask_grid = strands_above_cells_gpu(
            strand_positions, grid_pos, grid_normal, half_size,
            h_min=-0.01, h_max=0.4, grid_valid=grid_valid, show_progress=True,
            grid_t1=grid_t1, grid_t2=grid_t2,
        )

    # is_c1_mask 只跟這一輪迭代開始時的 strand_mask 有關，跟 (u_idx, v_idx) 無關，
    # 提到迴圈外算一次就好 (原本 update_color 裡是每格重算一次，等價但多做了 n_rows*n_cols 次)
    is_c1_mask = ~strand_mask  # False = base color = c1

    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]  # (K,) 這一格上方的髮絲索引
        if strands_inside.size == 0:
            continue

        mask_in_cell = strand_mask[strands_inside]  # (K,) bool 這一格上方的髮絲 mask
        mask_mode = int(mask_in_cell.sum()) * 2 > mask_in_cell.size  # 眾數 (True 嚴格過半才是 True，平手取 False)

        if not mask_mode:  # 如果眾數是 False (base_color)，就不用改了
            continue

        grid_score = single_grid_score2(
            strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx], half_size, is_c1_mask,
            grid_t1[u_idx, v_idx], grid_t2[u_idx, v_idx],
        )

        if grid_score <= threshold:  # 兩類分得夠開，這一格不需要處理
            continue

        strand_mask_iter[strands_inside] = mask_mode
    return strand_mask_iter

In [ ]:
import numpy as np
from scipy import sparse
from numba import njit
from tqdm import tqdm
from evaluation import *


# Fiduccia-Mattheyses 更新 base_coloring_state (兩種都「不建 (S, S) adjacency」，只用 cell <-> strand 的 incidence)
#   mode="clique": 每格視為一個 clique，邊權重 = 兩根髮絲共用的格數；最小化 sum_c n_true[c] * n_false[c]
#   mode="cutnet": 每格視為一條 net (hypergraph)；最小化「同時含兩種顏色的格子數」 (FM 原始論文的 cut-net 目標)

def build_cell_incidence(strand_mask_grid):
    """(n_rows, n_cols, S) bool -> 只保留含 >= 2 根髮絲的格子 (單根的格子不可能混色，對兩種目標都沒貢獻)，
    回傳 (cell_ptr, cell_members, strand_ptr, strand_cells) 兩份 CSR 索引：格子 -> 髮絲 / 髮絲 -> 格子。
    記憶體只有 O(nnz)，nnz = 所有 (格子, 髮絲) 配對數。"""
    n_rows, n_cols, n_strands = strand_mask_grid.shape
    M = sparse.csr_matrix(strand_mask_grid.reshape(n_rows * n_cols, n_strands))
    return _incidence_from_csr(M)


def _incidence_from_csr(M):
    M = M[np.diff(M.indptr) >= 2].tocsr()
    Mt = M.T.tocsr()
    return (M.indptr.astype(np.int64), M.indices.astype(np.int32),
            Mt.indptr.astype(np.int64), Mt.indices.astype(np.int32))


@njit
def _b_insert(i, key, head, nxt, prv):
    h = head[key]
    nxt[i] = h
    prv[i] = -1
    if h >= 0:
        prv[h] = i
    head[key] = i


@njit
def _b_remove(i, key, head, nxt, prv):
    p = prv[i]
    n = nxt[i]
    if p >= 0:
        nxt[p] = n
    else:
        head[key] = n
    if n >= 0:
        prv[n] = p


@njit
def _bump(u, d, gain, off, W, state, head, nxt, prv, tops):
    s = 1 if state[u] else 0  # u 還沒被鎖定，pass 期間 state[u] 不變，所以 bucket 側別固定
    _b_remove(u, s * W + gain[u] + off, head, nxt, prv)
    gain[u] += d
    key = gain[u] + off
    _b_insert(u, s * W + key, head, nxt, prv)
    if key > tops[s]:
        tops[s] = key


@njit
def _fm_pass(state, cell_ptr, cell_members, strand_ptr, strand_cells, mode, max_gain, patience, lo, hi):
    """一個 FM pass (in-place 修改 state)，回傳這個 pass 實際採用的累積 gain。
    mode 0 = clique, 1 = cutnet。gain bucket 是 (head, nxt, prv) 三個 int 陣列做成的雙向鏈結串列，
    True / False 兩側各一組 bucket (head 的前後兩半)，這樣才能在 True 的總數被限制在 [lo, hi]
    時，只從「移動後仍合法」的那一側挑最大 gain，避免整個分類塌縮成同一類。"""
    S = state.shape[0]
    R = cell_ptr.shape[0] - 1
    n1 = np.zeros(R, np.int64)  # 每格目前 True 的髮絲數
    size = np.empty(R, np.int64)
    for c in range(R):
        size[c] = cell_ptr[c + 1] - cell_ptr[c]
        for k in range(cell_ptr[c], cell_ptr[c + 1]):
            if state[cell_members[k]]:
                n1[c] += 1

    gain = np.zeros(S, np.int64)
    for v in range(S):
        g = 0
        for k in range(strand_ptr[v], strand_ptr[v + 1]):
            c = strand_cells[k]
            nF = n1[c] if state[v] else size[c] - n1[c]  # v 這一側 (含 v) 的髮絲數
            nT = size[c] - nF
            if mode == 0:
                g += nT - (nF - 1)
            else:
                if nF == 1:
                    g += 1
                if nT == 0:
                    g -= 1
        gain[v] = g

    off = max_gain
    W = 2 * max_gain + 1
    head = np.full(2 * W, -1, np.int64)
    nxt = np.empty(S, np.int64)
    prv = np.empty(S, np.int64)
    n_true = 0
    for v in range(S):
        sd = 1 if state[v] else 0
        n_true += sd
        _b_insert(v, sd * W + gain[v] + off, head, nxt, prv)

    locked = np.zeros(S, np.bool_)
    moves = np.empty(S, np.int64)
    n_moves = 0
    cum = 0
    best = 0
    best_step = 0
    tops = np.full(2, 2 * max_gain, np.int64)

    while n_moves < S:
        best_key = -1
        best_side = -1
        for sd in range(2):
            if sd == 1 and n_true - 1 < lo:  # 移動 True 側會讓 True 總數低於下限
                continue
            if sd == 0 and n_true + 1 > hi:
                continue
            while tops[sd] >= 0 and head[sd * W + tops[sd]] < 0:
                tops[sd] -= 1
            if tops[sd] > best_key:
                best_key = tops[sd]
                best_side = sd
        if best_side < 0:
            break
        v = head[best_side * W + best_key]
        _b_remove(v, best_side * W + best_key, head, nxt, prv)
        locked[v] = True
        cum += gain[v]
        moves[n_moves] = v
        n_moves += 1
        n_true += -1 if best_side == 1 else 1
        if cum > best:
            best = cum
            best_step = n_moves

        sv = state[v]
        for k in range(strand_ptr[v], strand_ptr[v + 1]):
            c = strand_cells[k]
            a = cell_ptr[c]
            b = cell_ptr[c + 1]
            nF = n1[c] if sv else size[c] - n1[c]
            nT = size[c] - nF
            if mode == 0:
                for m in range(a, b):
                    u = cell_members[m]
                    if u == v or locked[u]:
                        continue
                    _bump(u, 2 if state[u] == sv else -2, gain, off, W, state, head, nxt, prv, tops)
            else:
                if nT == 0:  # 整格都在 v 這側 -> 移動 v 會讓其他人「離開 v 這側」變成有機會消除混色
                    for m in range(a, b):
                        u = cell_members[m]
                        if u != v and not locked[u]:
                            _bump(u, 1, gain, off, W, state, head, nxt, prv, tops)
                elif nT == 1:  # 對側唯一的那根，之後移過來就不再消除混色
                    for m in range(a, b):
                        u = cell_members[m]
                        if state[u] != sv:
                            if not locked[u]:
                                _bump(u, -1, gain, off, W, state, head, nxt, prv, tops)
                            break
            if sv:
                n1[c] -= 1
            else:
                n1[c] += 1
            if mode == 1:
                nF2 = nF - 1
                if nF2 == 0:  # v 走後原本這側空了 -> 其他人 (都在對側) 移過去不再有消除混色的機會
                    for m in range(a, b):
                        u = cell_members[m]
                        if u != v and not locked[u]:
                            _bump(u, -1, gain, off, W, state, head, nxt, prv, tops)
                elif nF2 == 1:  # 原本這側只剩一根 -> 它移走就能消除混色
                    for m in range(a, b):
                        u = cell_members[m]
                        if u != v and state[u] == sv:
                            if not locked[u]:
                                _bump(u, 1, gain, off, W, state, head, nxt, prv, tops)
                            break
        state[v] = not sv

        if n_moves - best_step >= patience:  # 連續 patience 步累積 gain 沒創新高，提前結束這個 pass
            break

    for i in range(best_step, n_moves):  # 回滾沒被最佳前綴採用的移動
        state[moves[i]] = not state[moves[i]]
    return best


def fm_objectives(incidence, state):
    """回傳 (混色格數 cut-net, clique cut = sum_c n_true * n_false)。"""
    cell_ptr, cell_members, _, _ = incidence
    n1 = np.add.reduceat(state[cell_members].astype(np.int64), cell_ptr[:-1])
    size = np.diff(cell_ptr)
    return int(((n1 > 0) & (n1 < size)).sum()), int((n1 * (size - n1)).sum())


def fm_refine(incidence, state, mode="cutnet", max_passes=20, patience=None, balance_tol=0.02, verbose=True):
    """FM 迭代更新 2-way 分類 state (bool, (S,))，回傳新的 state (不改動輸入)。
    每個 pass 依 gain 由大到小把髮絲各移動一次並鎖住，只採用累積 gain 最大的前綴，重複到沒有改善。
    patience: 連續多少步累積 gain 沒創新高就提前結束該 pass；None = 不提前結束 (完整 FM)。
    balance_tol: True 的總數只能偏離初始值 balance_tol * S 以內 (預設 2%)；None = 不限制。
      cut-net 目標「全部同一類」就是 0 混色，不設限制的話會塌縮成單一顏色，所以預設要限。"""
    cell_ptr, cell_members, strand_ptr, strand_cells = incidence
    n_strands = strand_ptr.shape[0] - 1
    assert state.shape == (n_strands,)
    state = state.astype(np.bool_).copy()

    if mode == "clique":
        mode_id = 0
        entry_strand = np.repeat(np.arange(n_strands), np.diff(strand_ptr))
        max_gain = int(np.bincount(entry_strand, weights=np.diff(cell_ptr)[strand_cells] - 1,
                                   minlength=n_strands).max())
    elif mode == "cutnet":
        mode_id = 1
        max_gain = int(np.diff(strand_ptr).max())
    else:
        raise ValueError(mode)
    max_gain = max(max_gain, 1)
    if patience is None:
        patience = n_strands + 1
    n_true0 = int(state.sum())
    tol = n_strands if balance_tol is None else int(balance_tol * n_strands)
    lo, hi = max(0, n_true0 - tol), min(n_strands, n_true0 + tol)

    if verbose:
        print(f"[FM/{mode}] start: mixed cells = {fm_objectives(incidence, state)[0]}, "
              f"clique cut = {fm_objectives(incidence, state)[1]}")
    for pass_idx in tqdm(range(max_passes)):
        gained = _fm_pass(state, cell_ptr, cell_members, strand_ptr, strand_cells,
                          mode_id, max_gain, patience, lo, hi)
        if verbose:
            print(f"[FM/{mode}] pass {pass_idx}: gain = {gained}, "
                  f"mixed cells = {fm_objectives(incidence, state)[0]}, "
                  f"clique cut = {fm_objectives(incidence, state)[1]}")
        if gained <= 0:
            break
    return state


In [ ]:

incidence = build_cell_incidence(strands_mask_grid)
print("cells (>=2 strands):", incidence[0].shape[0] - 1, " incidence nnz:", incidence[1].shape[0])

# True = highlight (cell 19 塗色的 highlight 區塊)，False = base_color。mask 本身就是 bool，直接用。
base_coloring_state_init = base_strand_mask.copy()

state_clique = fm_refine(incidence, base_coloring_state_init, mode="clique")
state_cutnet = fm_refine(incidence, base_coloring_state_init, mode="cutnet")

base_coloring_state = state_cutnet  # 預設採用 cut-net 版本 (目標 = 每格顏色一致)
for name, st in [("init", base_coloring_state_init), ("clique", state_clique), ("cutnet", state_cutnet)]:
    print(f"{name:7s} True = {int(st.sum())}, False = {int((~st).sum())}, "
          f"changed vs init = {int((st != base_coloring_state_init).sum())}, "
          f"(mixed cells, clique cut) = {fm_objectives(incidence, st)}")

iter_strand_masks = [base_coloring_state_init, state_clique, state_cutnet]

In [ ]:
n_iter = 3

iter_strand_masks = [ base_strand_mask.copy() ]  # 每個元素 (S,) bool

for _ in range(n_iter):
    m_strand_mask = update_color(
        strand_positions, grid_pos, grid_normal, grid_valid,
        row_idx, col_idx, iter_strand_masks[-1], half_size,
        strand_mask_grid=strands_mask_grid,
        threshold=0.1,
        grid_t1=grid_t1, grid_t2=grid_t2,
    )
    iter_strand_masks.append(m_strand_mask)

In [ ]:
# 以髮根 UV 座標繪製髮絲顏色的 2D 點雲分布
# 每個點代表一根髮絲，顏色由 strand mask 經 mask_to_rgb 換算
n_plots = len(iter_strand_masks)
fig, axes = plt.subplots(n_plots, 1, figsize=(16, 4 * n_plots))
fig.set_facecolor("#1e1e1e")
for ax in axes:
    ax.set_facecolor("gray")    # 繪圖區域的底色（scatter/imshow 底下那層）

for i, strand_mask in enumerate(iter_strand_masks):
    draw_strand_uv_color_distribution(root_uv, mask_to_rgb(strand_mask), ax=axes[i], show=False)

plt.show()


In [ ]:
from tqdm import tqdm
from scipy.spatial import ConvexHull
def cal_grid_score(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, strand_mask, half_size,
    strand_mask_grid = None,
    *, grid_t1, grid_t2,  # (N, N, 3) 每格切線基底 (必填)，用 build_scalp_grid_tangents 產生
):
    n_rows, n_cols = grid_valid.shape

    if strand_mask_grid is None:
        strand_mask_grid = strands_above_cells_gpu(
            strand_positions, grid_pos, grid_normal, half_size,
            h_min=-0.01, h_max=0.4, grid_valid=grid_valid, show_progress=True,
            grid_t1=grid_t1, grid_t2=grid_t2,
        )
    is_c1_mask = ~strand_mask  # False = base color = c1

    grid_score = np.zeros((n_rows, n_cols), dtype=np.float32)  # (N, N) 每格的髮絲顏色熵值 (越大 = 顏色越分散)
    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        row, col = (n_cols - 1) - v_idx, u_idx  # (u_idx, v_idx) -> (row, col)，見上面說明

        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]  # (K,) 這一格上方的髮絲索引
        grid_score[row, col] = single_grid_score2(
            strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx], half_size, is_c1_mask,
            grid_t1[u_idx, v_idx], grid_t2[u_idx, v_idx],
        )
    return grid_score
for ax in axes:
    ax.set_facecolor("gray")    # 繪圖區域的底色（scatter/imshow 底下那層）

for i in range(n_iter+1):
    print(f"=== iteration {i} ===")
    grid_score = cal_grid_score(
        strand_positions, grid_pos, grid_normal, grid_valid,
        row_idx, col_idx, iter_strand_masks[i], half_size,
        strand_mask_grid=strands_mask_grid,
        grid_t1=grid_t1, grid_t2=grid_t2,
    )

    grid_score[grid_score < 0.99] = 0.0

    print("average grid score =", grid_score.mean())

    if(i):
        changed = iter_strand_masks[i] != iter_strand_masks[i-1]
        print(changed.shape)
        print("diff:", changed.mean())  # 這一輪被翻轉的髮絲比例

    plt.subplot(2, (n_iter+1)//2+1, i+1)

    plt.imshow(grid_score, origin="upper", cmap="magma", vmin=0, vmax=1)
    plt.colorbar(label="交點密度 (越大 =  顏色越分散)")
    plt.title(f"grid score (iteration {i})")

plt.show()

### 2D rendering result

In [ ]:
from tqdm import tqdm
from scipy.spatial import ConvexHull

def cal_2d_grid_color(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, strand_mask, half_size,
    strand_mask_grid
):
    n_rows, n_cols = grid_valid.shape

    # if strand_mask_grid is None:
    #     strand_mask_grid = strands_above_cells_gpu(
    #         strand_positions, grid_pos, grid_normal, half_size,
    #         h_min=-0.01, h_max=0.4, grid_valid=grid_valid, show_progress=True,
    #     )

    grid_mask = np.zeros((n_rows, n_cols), dtype=bool)  # (N, N) bool 每格的 mask 眾數
    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        row, col = (n_cols - 1) - v_idx, u_idx  # (u_idx, v_idx) -> (row, col)，見上面說明

        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]  # (K,) 這一格上方的髮絲索引
        # grid_score[row, col] = single_grid_score2(
        m = strand_mask[strands_inside]
        grid_mask[row, col] = m.sum() * 2 > m.size  # bool 的 median = 眾數 (平手 / 空格取 False)
        #     strands_inside, strand_positions, grid_pos[u_idx, v_idx], grid_normal[u_idx, v_idx], half_size, is_c1_mask
        # )
    return grid_mask


plt.figure(figsize=(6*(n_iter+1), 12 ))

for i in range(n_iter+1):
    print(f"=== iteration {i} ===")
    grid_mask_i = cal_2d_grid_color(
        strand_positions, grid_pos, grid_normal, grid_valid,
        row_idx, col_idx, iter_strand_masks[i], half_size,
        strand_mask_grid=strands_mask_grid
    )

    print(grid_mask_i.shape, grid_mask_i.dtype)

    plt.subplot(2, (n_iter+1)//2+1, i+1)

    plt.imshow(mask_to_rgb(grid_mask_i))
    # plt.imshow(grid_mask_i, origin="upper", cmap="magma", vmin=0, vmax=1)
    # plt.colorbar(label="交點密度 (越大 =  顏色越分散)")
    plt.title(f"grid mask (iteration {i})")

plt.show()

In [ ]:
## analyze result

show_direction_vec = True
show_edge_detect_score = False

from tqdm import tqdm
from scipy.spatial import ConvexHull

def cal_2d_grid_color(
    strand_positions, grid_pos, grid_normal, grid_valid,
    row_idx, col_idx, strand_mask, half_size,
    strand_mask_grid
):
    n_rows, n_cols = grid_valid.shape

    grid_mask = np.zeros((n_rows, n_cols), dtype=bool)  # (N, N) bool 每格的 mask 眾數
    for u_idx, v_idx in tqdm(np.ndindex(n_rows, n_cols), total=n_rows * n_cols):
        row, col = (n_cols - 1) - v_idx, u_idx  # (u_idx, v_idx) -> (row, col)，見上面說明

        strands_inside = np.where(strand_mask_grid[u_idx, v_idx])[0]  # (K,) 這一格上方的髮絲索引
        m = strand_mask[strands_inside]
        grid_mask[row, col] = m.sum() * 2 > m.size  # 眾數 (平手 / 空格取 False)
    return grid_mask


# --- show_direction_vec / show_edge_detect_score 疊圖用的縮小顯示參數 ---
# 箭頭/通量 marker 都刻意縮小 (短箭頭、細線、小點)，疊在 grid mask 上時才不會蓋住底下的顏色。
quiver_scale = 60                # quiver 的 scale 參數，越大箭頭畫出來越短
quiver_width = 0.0015            # 箭頭線寬 (縮小顯示體積)
quiver_alpha = 0.5                # 箭頭透明度：未平均後箭頭數 = 髮絲數，半透明避免疊起來糊成一片
max_arrows = 6000                # 箭頭數上限，超過就固定隨機種子做子取樣 (仍是未平均的原始向量)
score_marker_max = 14            # edge score 疊加點的最大 marker 面積 (縮小顯示體積，不畫滿整格)
score_alpha = 0.85                # edge score 疊加點的透明度，留一點底下顏色透出來

plt.figure(figsize=(8 ,8*(n_iter+1)))

for i in range(n_iter+1):
    print(f"=== iteration {i} ===")
    grid_mask_i = cal_2d_grid_color(
        strand_positions, grid_pos, grid_normal, grid_valid,
        row_idx, col_idx, iter_strand_masks[i], half_size,
        strand_mask_grid=strands_mask_grid
    )

    ax = plt.subplot(n_iter+1, 1, i+1)

    ax.set_facecolor("gray")  # 繪圖區域的底色（scatter/imshow 底下那層）
    ax.imshow(mask_to_rgb(grid_mask_i))

    if show_direction_vec:
        # 跟前面「每根髮絲的向量」同一套場 (cell_flow_vectors / compute_strand_vectors_2d)，
        # 不做 grid 內平均，保留每根髮絲自己的取樣點與流向；箭頭故意縮小、半透明、並隨機
        # 子取樣，避免整張圖被箭頭蓋住看不到底下的髮色。
        strand_pts2d_i, strand_vec2d_i = compute_strand_vectors_2d(
            strand_positions, grid_pos, grid_normal, grid_valid, half_size,
            iter_strand_masks[i], base_color, strands_mask_grid, grid_t1, grid_t2,
        )
        if strand_pts2d_i.shape[0] > max_arrows:
            rng = np.random.default_rng(i)
            pick = rng.choice(strand_pts2d_i.shape[0], size=max_arrows, replace=False)
            strand_pts2d_i, strand_vec2d_i = strand_pts2d_i[pick], strand_vec2d_i[pick]
        ax.quiver(
            strand_pts2d_i[:, 0], strand_pts2d_i[:, 1],
            strand_vec2d_i[:, 0], strand_vec2d_i[:, 1],
            color="cyan", scale=quiver_scale, width=quiver_width, alpha=quiver_alpha,
        )

    if show_edge_detect_score:
        # 跟「勾邊測試」同一套 edge_detect_score (3D 向量場通量)，用小 marker 疊加而不是整格塗滿色塊；
            # marker 面積/透明度都跟著分數走，分數趨近 0 的格子幾乎看不到 marker，不會蓋住底色。
        grid_score_i = cal_grid_edge_score(
            strand_positions, grid_pos, grid_normal, grid_valid,
            row_idx, col_idx, iter_strand_masks[i], half_size, base_color,
            strand_mask_grid=strands_mask_grid,
            grid_t1=grid_t1, grid_t2=grid_t2,
        )
        score_row_idx, score_col_idx = np.nonzero(grid_score_i > 1e-3)
        score_vals = np.clip(grid_score_i[score_row_idx, score_col_idx], 0, 1)
        ax.scatter(
            score_col_idx, score_row_idx,
            # marker 面積 (scatter 的 s) 跟半徑的平方成正比，所以要讓「半徑」隨分數線性縮小、
            # 最大分數時維持 score_marker_max 這個大小，s 要跟 score_vals**2 成正比 (而不是 score_vals)：
            # s = score_marker_max 時 radius = R_max；score_vals 變成一半時 radius 也變成一半，
            # 對應的 s (= π r^2) 就要變成 1/4，也就是 score_vals**2 的縮放。
            s=(score_vals ** 2) * score_marker_max, c=score_vals,
            cmap="magma", vmin=0, vmax=1, alpha=score_alpha, linewidths=0,
        )

    ax.set_xlim(-0.5, N - 0.5)
    ax.set_ylim(N - 0.5, -0.5)  # 跟 imshow(origin="upper") 同方向: row 由上往下增加
    ax.set_title(f"grid mask (iteration {i})")

plt.show()

In [ ]:
# =====================================================================================
# Active Contour 測試 (3D 版)：snake 直接在 3D 空間 (頭皮曲面) 上跑
#
# 觀念 —— 髮絲流向場是頭皮本身的性質，場的不連續處 (分邊線 / 髮旋，也就是 |flux| 的山脊) 就是天然最佳的
# 顏色邊界：沿著它切，髮絲不會被切成兩色。做法：
#   1. 每格的 3D 淨向外通量 (cal_grid_flux_3d) -> |flux| 邊緣圖 -> Hessian 山脊偵測 + 遲滯門檻 + 長度過濾 -> 自然邊界 (格子集合)
#   2. 自然邊界的每一格、以及 base mask 的初始輪廓，都用該格的 3D 位置 (頭皮曲面) 表示；
#      contour 是一串 3D 點，snake 的所有量都在 3D 世界座標量測:
#        * 吸附: 3D 歐式距離下最近的自然邊界點 (cKDTree)，snap_radius 也是 3D 長度 (= snap_radius 個 cell 間距)
#        * 錨定: 初始 3D 輪廓上最近的點
#        * 內部能量 (彈性 / 剛性): 以 3D 弧長等距重採樣 (spacing = 1 個 cell 間距)
#      每次更新後把 3D 點投影回頭皮曲面 (Gauss-Newton 找曲面上最近點)，snake 不會飄離頭皮
#   3. 只有最後輸出 N x N 的 boolean mask 時，才把 3D 輪廓對應回格子座標 (row, col) 做 even-odd 填色；畫圖同理
#
# 流程：每格 3D 通量 -> |flux| 邊緣圖 -> 自然邊界 -> 3D snake -> (投影回格子) 新 mask
# 輸出：ac_mask (N, N) bool；ac_strand_mask (S,) bool (跟 iter_strand_masks 的元素同格式，可直接餵給後面的流程)
# =====================================================================================

import cv2
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from matplotlib.path import Path


# ---------- 1. 邊緣圖: 每格 3D 通量 -> |flux|；(2D 稠密流向場只用於視覺化) ----------
def rasterize_flow_field(pts2d, vec2d, n, sigma=1.0, min_count=1.0):
    """**只用於視覺化**：把「勾邊測試」算出的逐髮絲 (取樣位置, 3D 流向投影到 2D 後的向量) 點雲，用 normalized convolution 鋪成 (n, n) 稠密向量場。

    pts2d 已經是 (col, row) 圖像座標 (格子 (row, col) 的中心 = 整數座標)，vec2d 的 y 分量也已對齊 row 方向。
    向量先正規化成單位方向 (只看流向、不受髮絲投影長度影響)，再累加進所在格子，最後對「向量總和」與「髮絲數」
    各做一次 Gaussian 平滑後相除 = 局部平均流向。局部方向雜亂 / 相反的地方平均後長度 < 1，
    完全沒有髮絲的地方 (count 太低) 設為 0。回傳 vx, vy (皆為 (n, n))，x = col 方向, y = row 方向。
    """
    unit = vec2d / (np.linalg.norm(vec2d, axis=1, keepdims=True) + 1e-9)
    col = np.clip(np.rint(pts2d[:, 0]).astype(int), 0, n - 1)
    row = np.clip(np.rint(pts2d[:, 1]).astype(int), 0, n - 1)

    sum_x, sum_y, count = (np.zeros((n, n), np.float64) for _ in range(3))
    np.add.at(sum_x, (row, col), unit[:, 0])
    np.add.at(sum_y, (row, col), unit[:, 1])
    np.add.at(count, (row, col), 1.0)

    cnt_s = ndi.gaussian_filter(count, sigma)
    ok = cnt_s > min_count * 1e-3 * count.sum() / (n * n)  # 相對於平均密度的極小門檻: 只排除真正沒髮絲的格子
    vx = np.where(ok, ndi.gaussian_filter(sum_x, sigma) / np.maximum(cnt_s, 1e-9), 0.0)
    vy = np.where(ok, ndi.gaussian_filter(sum_y, sigma) / np.maximum(cnt_s, 1e-9), 0.0)
    return vx, vy, ok


def flow_edge_map(flux, valid, smooth=1.0, pct=98):
    """邊緣圖 f = |flux|。flux 是每格直接在 3D 空間算出來的向量場淨向外通量 (cal_grid_flux_3d，(row, col) 排列)，
    不是由 2D 稠密場再用 np.gradient 求導。在頭皮有效範圍內用第 pct 百分位正規化到 [0, 1]。"""
    f = ndi.gaussian_filter(np.abs(flux), smooth)
    f[~valid] = 0.0
    return np.clip(f / (np.percentile(f[valid], pct) + 1e-9), 0, 1)


# ---------- 2. 抽出向量場的「自然邊界」: |flux| 的山脊線 ----------
def extract_natural_boundary(edge, valid, sigma=1.0, t_low=0.5, t_high=0.85, min_extent=15):
    """Steger 式山脊偵測: 用 Hessian 最負特徵值的方向當「跨山脊方向」，沿該方向做非極大值抑制，
    只留山脊中心線；再做遲滯門檻 + 長度過濾:
      * 連通元件內至少要有一點 edge >= t_high (真的很強的邊界)，元件裡其餘點 edge >= t_low 即可跟著保留
      * 元件的外接框最長邊 >= min_extent (cell)，濾掉孤立的點狀峰值 (髮旋中心之類的局部高通量，不是一條線)
    回傳 (n, n) bool 自然邊界 mask。"""
    hyy = ndi.gaussian_filter(edge, sigma, order=(2, 0))
    hxx = ndi.gaussian_filter(edge, sigma, order=(0, 2))
    hxy = ndi.gaussian_filter(edge, sigma, order=(1, 1))
    tr, det = hxx + hyy, hxx * hyy - hxy ** 2
    lam = tr / 2 - np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))  # 最負特徵值 = 跨山脊方向的曲率
    nx, ny = hxy, lam - hxx                                   # 對應的特徵向量 (x = col, y = row)
    norm = np.hypot(nx, ny)
    bad = norm < 1e-9
    nx = np.where(bad, 1.0, nx / np.where(bad, 1, norm))
    ny = np.where(bad, 0.0, ny / np.where(bad, 1, norm))

    rr, cc = np.mgrid[0:edge.shape[0], 0:edge.shape[1]].astype(float)
    f_p = ndi.map_coordinates(edge, [rr + ny, cc + nx], order=1, mode="nearest")
    f_m = ndi.map_coordinates(edge, [rr - ny, cc - nx], order=1, mode="nearest")
    ridge = (edge >= f_p) & (edge >= f_m) & (lam < 0) & (edge > t_low) & valid

    lab, n_lab = ndi.label(ridge, structure=np.ones((3, 3)))
    keep = np.zeros(n_lab + 1, dtype=bool)
    for i, sl in enumerate(ndi.find_objects(lab), start=1):
        comp = lab[sl] == i
        extent = max(sl[0].stop - sl[0].start, sl[1].stop - sl[1].start)
        keep[i] = extent >= min_extent and edge[sl][comp].max() >= t_high
    return keep[lab]


# ---------- 3. 頭皮曲面 (3D) 與格子座標 (x=col, y=row) 之間的對應 ----------
def build_surface_rc(grid_pos, grid_valid):
    """(u_idx, v_idx) 排列的 grid_pos -> (row, col) 排列 (row = N-1-v_idx, col = u_idx，跟 mask_matrix 一致)，
    無效格 (grid_pos = 0) 用最近的有效格填補，讓 bilinear 內插到頭皮邊緣時不會被拉向原點。回傳 (N, N, 3)。"""
    pos_rc = grid_pos.transpose(1, 0, 2)[::-1].astype(np.float64)
    valid = grid_valid.T[::-1, :]
    _, (ir, ic) = ndi.distance_transform_edt(~valid, return_indices=True)
    return pos_rc[ir, ic]


def surface_point(surf, xy):
    """格子座標 xy (m, 2) = (x=col, y=row) -> 頭皮曲面上的 3D 點 (m, 3)，對 surf 做 bilinear 內插。"""
    q = [xy[:, 1], xy[:, 0]]
    return np.stack([ndi.map_coordinates(surf[..., d], q, order=1, mode="nearest") for d in range(3)], axis=1)


def project_to_surface(pts, surf, surf_tree, n_newton=5):
    """把任意 3D 點投影回頭皮曲面。先用 KDTree 找最近的格子中心當初值，再對 bilinear 曲面 X(col, row) 做 Gauss-Newton
    最小化 |X - p|^2。回傳 (曲面上的 3D 點 (m, 3), 對應的格子座標 xy (m, 2))。"""
    n_rows, n_cols = surf.shape[:2]
    _, k = surf_tree.query(pts)
    xy = np.stack([k % n_cols, k // n_cols], axis=1).astype(np.float64)
    for _ in range(n_newton):
        x = surface_point(surf, xy)
        x_col = (surface_point(surf, xy + [1, 0]) - surface_point(surf, xy - [1, 0])) / 2  # dX/dcol
        x_row = (surface_point(surf, xy + [0, 1]) - surface_point(surf, xy - [0, 1])) / 2  # dX/drow
        jac = np.stack([x_col, x_row], axis=2)                                              # (m, 3, 2)
        jtj = jac.transpose(0, 2, 1) @ jac + 1e-12 * np.eye(2)
        jtr = np.einsum("mij,mi->mj", jac, pts - x)
        step = np.linalg.solve(jtj, jtr[..., None])[..., 0]
        xy = np.stack([np.clip(xy[:, 0] + np.clip(step[:, 0], -1.5, 1.5), 0, n_cols - 1),
                       np.clip(xy[:, 1] + np.clip(step[:, 1], -1.5, 1.5), 0, n_rows - 1)], axis=1)
    return surface_point(surf, xy), xy


# ---------- 4. Snake (Kass et al.) 直接在 3D 空間: 以 base mask 的邊界 (提升到 3D) 為初始 contour ----------
def _resample_closed(pts, spacing=1.0):
    """封閉折線依弧長等距重採樣，pts (m, D)，D = 2 或 3 都可以；spacing 的單位跟 pts 一致。"""
    seg = np.linalg.norm(np.roll(pts, -1, axis=0) - pts, axis=1)
    total = seg.sum()
    m = max(int(round(total / spacing)), 8)
    s = np.concatenate([[0], np.cumsum(seg)])
    t = np.linspace(0, total, m, endpoint=False)
    closed = np.vstack([pts, pts[:1]])
    return np.stack([np.interp(t, s, closed[:, d]) for d in range(pts.shape[1])], axis=1)


def _snake_matrix_inv(m, alpha, beta, gamma):
    """(A + gamma I)^-1，A = beta * D4 - alpha * D2 (closed 環狀差分)，pentadiagonal cyclic。"""
    a = np.zeros((m, m))
    idx = np.arange(m)
    a[idx, idx] = 2 * alpha + 6 * beta + gamma
    for k, w in ((1, -(alpha + 4 * beta)), (2, beta)):
        a[idx, (idx + k) % m] += w
        a[idx, (idx - k) % m] += w
    return np.linalg.inv(a)


def boundary_snake_3d(contour3d, boundary_tree, surf, surf_tree, spacing, snap_radius,
                      alpha=0.2, beta=0.4, gamma=1.0, kappa=1.0, anchor=0.5,
                      n_iter=100, resample_every=5):
    """單一封閉 3D contour (m, 3)。每個點受兩種外力 (都是 3D 位移，單位 = 世界座標長度):
      * 吸附力 kappa * w * (最近自然邊界點 - 目前位置): 距離 <= snap_radius 時 w = 1 (完全吸附)，
        線性降到 2 * snap_radius 處 w = 0；boundary_tree = None (沒有自然邊界) 時 w = 0，輪廓不動
      * 錨定力 anchor * (1 - w) * (初始輪廓上最近的點 - 目前位置): 沒有邊界可吸的地方，別讓 snake 亂飄
    內部能量 (alpha 彈性 / beta 剛性) 以 3D 弧長等距 (spacing) 的點列計算，讓吸附後的輪廓保持平滑，吸附段與
    沒吸附的段之間也靠它順接。每次更新後投影回頭皮曲面。回傳 (演化後的 3D 輪廓, 對應的格子座標 xy)。"""
    pts = _resample_closed(contour3d, spacing)
    dense = _resample_closed(contour3d, spacing / 4)
    tree = cKDTree(dense)
    boundary_pts = boundary_tree.data if boundary_tree is not None else None
    ainv, ainv_m = None, -1
    for it in range(n_iter):
        if it % resample_every == 0:
            pts = _resample_closed(pts, spacing)
        m = pts.shape[0]
        if m != ainv_m:
            ainv, ainv_m = _snake_matrix_inv(m, alpha, beta, gamma), m
        if boundary_tree is not None:
            dist, j = boundary_tree.query(pts)
            w = np.clip(2.0 - dist / snap_radius, 0.0, 1.0)
            snap = (boundary_pts[j] - pts) * w[:, None]
        else:
            w, snap = np.zeros(m), np.zeros_like(pts)
        _, j = tree.query(pts)
        hold = (dense[j] - pts) * (anchor * (1 - w))[:, None]
        pts = ainv @ (gamma * pts + kappa * snap + hold)
        pts, xy = project_to_surface(pts, surf, surf_tree)
    return pts, xy


# ---------- 5. 對 base mask 跑 3D active contour ----------
def mask_to_contours(mask, up=4):
    """boolean (n, n) mask -> 封閉輪廓列表 (每個 (m, 2), 座標 x=col, y=row，單位=cell)。
    先把 mask 放大 up 倍再取輪廓 (輪廓落在放大後像素中心，放大越多越貼近真正的格線邊界)，再換算回 cell 座標。"""
    big = np.kron(mask.astype(np.uint8), np.ones((up, up), np.uint8))
    big = np.pad(big, 1)  # 貼著邊界的 mask 也要能形成封閉輪廓
    found, _ = cv2.findContours(big, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    return [((c[:, 0, :].astype(np.float64) - 1 + 0.5) / up - 0.5) for c in found if len(c) >= 8]


def contours_to_mask(contours, shape):
    """封閉輪廓 (格子座標 x=col, y=row) -> boolean mask (cell 中心落在輪廓內即為 True)；用 even-odd (XOR) 規則，內孔自然挖空。"""
    rr, cc = np.mgrid[0:shape[0], 0:shape[1]]
    centers = np.stack([cc.ravel(), rr.ravel()], axis=1)
    out = np.zeros(shape[0] * shape[1], dtype=bool)
    for c in contours:
        out ^= Path(c).contains_points(centers)
    return out.reshape(shape)


def active_contour_mask_3d(base_mask, boundary_tree, surf, surf_tree, spacing, snap_radius, **snake_kwargs):
    """以 base_mask 的邊界為初始輪廓 (先提升到 3D 頭皮曲面)，在 3D 空間吸附到自然邊界。
    只有最後才把 3D 輪廓對應回格子座標 (row, col) 填成 (N, N) boolean mask。
    回傳 (新 mask, 初始 3D 輪廓列表, 演化後 3D 輪廓列表, 初始輪廓的格子座標列表, 演化後輪廓的格子座標列表)；
    後兩者只用於視覺化 / 填 mask。"""
    init_xy = mask_to_contours(base_mask)
    init3d = [surface_point(surf, c) for c in init_xy]
    final3d, final_xy = [], []
    for c3 in init3d:
        p3, xy = boundary_snake_3d(c3, boundary_tree, surf, surf_tree, spacing, snap_radius, **snake_kwargs)
        final3d.append(p3)
        final_xy.append(xy)
    return contours_to_mask(final_xy, base_mask.shape), init3d, final3d, init_xy, final_xy



# ---------------------------------- 執行 ----------------------------------
flow_sigma = 1.5     # (只影響視覺化的) 2D 稠密流向場的 Gaussian 平滑 (cell)
edge_smooth = 1.5    # 邊緣圖 |flux| 的平滑 (cell)
snap_radius = 8      # 輪廓離自然邊界多近 (單位: cell 間距，換算成 3D 長度) 才會被吸附上去；越大越容易被遠處的邊界拉走
snake_kwargs = dict(
    alpha=0.2,       # 彈性 (拉緊 contour)
    beta=0.4,        # 剛性 (抗彎曲)
    kappa=1.0,       # 吸附強度，1 = 一步到位貼上自然邊界
    anchor=0.5,      # 沒有邊界可吸的輪廓段固定在原位的強度
    n_iter=100,
)

# grid_valid 是 (u_idx, v_idx) 排列，換成跟 mask_matrix 一致的 (row, col) 排列: row = (N-1) - v_idx, col = u_idx
valid_rc = grid_valid.T[::-1, :]

# 3D 淨向外通量 (直接在 3D 局部座標算，跟 2D 投影無關) -> 邊緣圖 -> 自然邊界 (格子集合)
flux3d, flux_ok = cal_grid_flux_3d(
    strand_positions, grid_pos, grid_normal, grid_valid, half_size, strands_mask_grid,
    grid_t1=grid_t1, grid_t2=grid_t2,
)
edge_map = flow_edge_map(flux3d, valid_rc & flux_ok, smooth=edge_smooth)
natural_boundary = extract_natural_boundary(edge_map, ndi.binary_erosion(valid_rc & flux_ok, iterations=3))  # 頭皮外緣會有假邊界，先內縮排除

# 2D 稠密流向場: 只用來畫圖 (把 3D 流向投影到 (t1, t2) 平面後鋪成稠密場)
flow_vx, flow_vy, flow_ok = rasterize_flow_field(strand_pts2d, strand_vec2d, N, sigma=flow_sigma)

# 3D 頭皮曲面與自然邊界的 3D 點集
surf_rc = build_surface_rc(grid_pos, grid_valid)                 # (N, N, 3)，(row, col) 排列
surf_tree = cKDTree(surf_rc.reshape(-1, 3))                      # flat index = row * N + col
cell_spacing = 2 * half_size                                     # 相鄰格子的 3D 間距 (estimate_half_size 的定義)
boundary_pts3d = surf_rc[natural_boundary]                       # (B, 3) 自然邊界每一格在頭皮上的 3D 位置
boundary_tree = cKDTree(boundary_pts3d) if len(boundary_pts3d) else None
print(f"cell 間距 = {cell_spacing:.5f}, 自然邊界 {len(boundary_pts3d)} 格, snap_radius = {snap_radius * cell_spacing:.5f} (3D 長度)")


n_iter = 3
for _ in range(n_iter):
    ac_mask, init3d, final3d, init_contours, final_contours = active_contour_mask_3d(
        mask_matrix, boundary_tree, surf_rc, surf_tree, cell_spacing, snap_radius * cell_spacing, **snake_kwargs
    )
    ac_strand_mask = grid_mask_to_strand_mask(ac_mask, row_idx, col_idx)  # (S,) bool: 每根髮絲自己的 mask，跟 base_strand_mask 同樣的查表方式

    print("base mask 面積 :", int(mask_matrix.sum()), "cells")
    print("AC   mask 面積 :", int(ac_mask.sum()), "cells")
    print("被翻轉的 cell   :", int((ac_mask != mask_matrix).sum()), "cells")
    print("被翻轉的髮絲比例:", float((ac_strand_mask != base_strand_mask).mean()))


    # ---------------------------------- 視覺化 (3D 輪廓投影回格子座標，只用於畫圖) ----------------------------------
    def draw_contours(ax, contours, **kw):
        for c in contours:
            ax.plot(*np.vstack([c, c[:1]]).T, **kw)
    plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'Droid Sans Fallback', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False


    step = 3
    gx, gy = np.meshgrid(np.arange(0, N, step), np.arange(0, N, step))
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    axes[0].imshow(np.where(valid_rc, 0.85, 0.3), cmap="gray", vmin=0, vmax=1)
    axes[0].quiver(gx, gy, flow_vx[::step, ::step], flow_vy[::step, ::step],
                color="tab:blue", angles="xy", scale_units="xy", scale=0.5, width=0.003)
    axes[0].set_title("稠密流向場 (3D 流向投影到 2D 的單位向量局部平均，僅視覺化)")

    im = axes[1].imshow(edge_map, cmap="magma", vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1], fraction=0.046, label="3D |flux| (正規化)")
    by, bx = np.nonzero(natural_boundary)
    axes[1].scatter(bx, by, s=5, c="cyan", label="自然邊界")
    draw_contours(axes[1], init_contours, color="white", ls="--", lw=1.2)
    draw_contours(axes[1], final_contours, color="lime", lw=1.8)
    axes[1].set_title("邊緣圖 + 自然邊界 (cyan)\n白虛線 = 初始輪廓，綠線 = 3D 演化後輪廓 (投影回格子)")

    axes[2].imshow(mask_to_rgb(mask_matrix))
    draw_contours(axes[2], init_contours, color="tab:red", lw=1.2)
    axes[2].set_title("base mask")

    axes[3].imshow(mask_to_rgb(ac_mask))
    draw_contours(axes[3], final_contours, color="tab:red", lw=1.2)
    axes[3].set_title("3D active contour 後的 mask")

    for ax in axes:
        ax.set_xlim(-0.5, N - 0.5)
        ax.set_ylim(N - 0.5, -0.5)  # 跟 imshow(origin="upper") 同方向: row 由上往下增加
    plt.show()

    # 3D 視圖: 頭皮 (灰 = base color、深灰 = highlight) + 自然邊界 + 初始 / 演化後的 3D 輪廓
    fig3 = plt.figure(figsize=(7, 7))
    ax3 = fig3.add_subplot(111, projection="3d")
    vp, vm = surf_rc[valid_rc], mask_matrix[valid_rc]
    ax3.scatter(vp[:, 0], vp[:, 2], vp[:, 1], s=3, c=np.where(vm, 0.25, 0.8), cmap="gray", vmin=0, vmax=1, alpha=0.4)
    if len(boundary_pts3d):
        ax3.scatter(boundary_pts3d[:, 0], boundary_pts3d[:, 2], boundary_pts3d[:, 1], s=8, c="cyan", label="自然邊界")
    for c3 in init3d:
        c3 = np.vstack([c3, c3[:1]]); ax3.plot(c3[:, 0], c3[:, 2], c3[:, 1], color="tab:red", lw=1.2)
    for c3 in final3d:
        c3 = np.vstack([c3, c3[:1]]); ax3.plot(c3[:, 0], c3[:, 2], c3[:, 1], color="lime", lw=2)
    ax3.set_xlabel("x"); ax3.set_ylabel("z"); ax3.set_zlabel("y")
    ax3.set_title("3D active contour (紅 = 初始輪廓，綠 = 演化後輪廓)")
    ax3.legend()
    plt.show()

    # init_contours = final_contours
    mask_matrix = ac_mask

iter_strand_masks = [ base_strand_mask, grid_mask_to_strand_mask(ac_mask, row_idx, col_idx) ]  # 都是 (S,) 逐髮絲 mask
n_iter = 1

## 6. 存成 `coloring_by_strand` 的 per-strand template 格式，並用 Blender 渲染正面照

`coloring_by_strand/generate_highlight_render_by_strand.py` 這條 pipeline 認得的 per-strand
template，就是 `generate_highlight_templates_by_strand.py` 產生的格式，一個 npz 內含：

| key | 說明 |
|---|---|
| `highlighted` | `(nr_strands,)` bool，這根髮絲要不要套用 `highlight_color` |
| `highlight_color` | `(nr_strands, 3)` float32，`[0,1]` 的 RGB — 每根髮絲自己的顏色 |
| `root_uv` | `(nr_strands, 2)` float32 — 這個 template 是針對哪個 `root_uv` 排列烤出來的，渲染時會拿來跟 `--input_npz` 自己的 `root_uv` 核對，確保沒有對錯髮型/順序 |

跟舊版 `coloring_by_grid` 的 `{mask, grid_size, rgb}` 格式不同——這裡完全不需要 n×n grid，Blender
渲染端 (`generate_highlight_render_by_strand.py`) 也不會再做任何 UV 查表，直接把 `highlighted[k]` /
`highlight_color[k]` 逐一套用到第 `k` 根髮絲上。

把第 5 節算好的 `strand_highlighted` / strand mask（用 `mask_to_rgb` 換成 RGB 的 `strand_colors`）存成這個格式（`mask` 全開的等效寫法 = 全部
`highlighted=True`，整顆頭都交給查表結果決定顏色），再直接呼叫 Blender 端的
`coloring_by_strand/generate_highlight_render_by_strand.py`（不需要也沒有 `--use_template_color`
這種 flag——per-strand template 本來就是每根髮絲都各自帶著自己的顏色），渲染一張正面照。

In [ ]:
from IPython.display import Image, display
fig, ax = plt.subplots(2,1,figsize=(6*(n_iter+1), 6))
GEN3D_DIR = os.path.join(PROJECT_ROOT, "highlighting", "generate_from_3D_models")
CODE_BY_STRAND_DIR = os.path.join(GEN3D_DIR, "coloring_by_strand")

RENDER_SCRIPT = os.path.join(CODE_BY_STRAND_DIR, "generate_highlight_render_by_strand.py")
RUN_BLENDER_SH = os.path.join(CODE_BY_STRAND_DIR, "run_highlight_render.sh")
BLENDER_PATH = "/home/kyh/blender/blender"


fig, ax = plt.subplots(2,1,figsize=(6, 6))

# base_npz_path = save_template_npz(
#     template_dir=os.path.join(PROJECT_ROOT, "highlighting", "generate_from_3D_models", "coloring_by_strand", "base_templates_3D"),
#     sample_name=os.path.basename(HAIRSTYLE_DIR),
#     strands_source_name="full",
#     root_uv=root_uv,
#     strand_highlighted=strand_highlighted,
#     strand_colors=mask_to_rgb(base_strand_mask),  # bool mask -> RGB, Blender 端吃 RGB
#     grid_size=N,
#     color_source="mask_updating_test_3D.ipynb 第 5 節自訂的 mask_matrix",
#     source="mask_updating_test_3D.ipynb",
# )

# run_blender_render(
#     base_npz_path,
#     out_dir=os.path.join(HERE, "output", "base_3D"),
#     strands_npz=STRANDS_NPZ,
#     dataset_path=DATASET_PATH,
#     here=HERE,
#     run_blender_sh=RUN_BLENDER_SH,
#     blender_path=BLENDER_PATH,
#     render_script=RENDER_SCRIPT
# )


# display(Image(filename=os.path.join(HERE, "output", "base_3D", "front_multiview.png")))
for i in range(1, n_iter+1):
    strand_mask_i = iter_strand_masks[i]
    npz_path = save_template_npz(
        template_dir=os.path.join(PROJECT_ROOT, "highlighting", "generate_from_3D_models", "coloring_by_strand", "median_templates_3D"),
        sample_name=os.path.basename(HAIRSTYLE_DIR),
        strands_source_name="full",
        root_uv=root_uv,
        strand_highlighted=strand_highlighted,
        strand_colors=mask_to_rgb(strand_mask_i),
        grid_size=N,
        color_source="mask_updating_test_3D.ipynb 第 5 節自訂的 mask_matrix",
        source="mask_updating_test_3D.ipynb",
    )

    run_blender_render(
        npz_path,
        out_dir=os.path.join(HERE, "output", "iter_3D_" + str(i)),
        strands_npz=STRANDS_NPZ,
        dataset_path=DATASET_PATH,
        here=HERE,
        run_blender_sh=RUN_BLENDER_SH,
        blender_path=BLENDER_PATH,
        render_script=RENDER_SCRIPT,
    )